# Task 2: ALS Gene Perturbations with GeneFormer V2

## Objective
Apply the perturbation workflow from Task 1 to a curated panel of 
ALS-associated genes, in both healthy (PN) and diseased (sALS) cells.

## Dataset
Single-nucleus RNA-seq of the human primary motor cortex in ALS and FTLD patients.
- Source: GSE174332 (Pineda et al., Cell 2024)
- ~380,000 nuclei across 64 individuals

## ALS Gene Panel
Genes selected based on known ALS pathology (see config below):

| Gene | Role in ALS |
|---|---|
| SOD1 | First ALS gene identified; toxic protein aggregation |
| C9orf72 | Most common ALS mutation; hexanucleotide repeat expansion |
| TARDBP | Encodes TDP-43; aggregates in >95% of all ALS cases |
| FUS | RNA-binding protein; mislocalises in ALS |
| UBQLN2 | Ubiquitin pathway; protein clearance failure |
| OPTN | Autophagy receptor; loss of function in ALS |
| TBK1 | Innate immunity / autophagy; haploinsufficiency in ALS |
| VCP | Protein homeostasis; multisystem proteinopathy |
| HNRNPA1 | RNA-binding protein; prion-like domain aggregation |
| HNRNPA2B1 | RNA-binding protein; stress granule formation |
| NEK1 | DNA damage response; cytoskeletal integrity |
| SETX | RNA/DNA helicase; R-loop resolution |
| ANG | Angiogenin; neuroprotective RNA processing |
| PFN1 | Actin dynamics; axonal transport disruption |
| SQSTM1 | p62; autophagy cargo receptor |

### Imports

In [1]:
import sys
sys.path.append("..")

import logging
logging.basicConfig(level=logging.INFO)

import numpy as np
import scanpy as sc
import anndata as ad
import torch
from helical.models.geneformer import Geneformer, GeneformerConfig

from src.perturbation_v3 import (
    _clean_adata,
    _subsample_als,
    run_perturbation_pipeline,
    split_by_condition,
)

/Users/aseelawdeh/Documents/helical/als_perturbation/helical/lib/python3.11/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound
INFO:datasets:PyTorch version 2.7.0 available.
2026-03-10 10:04:18,353 - INFO:datasets:PyTorch version 2.7.0 available.


### Config

In [3]:
MODES          = ["knockdown", "knockup"]
CONDITION_COL  = "Condition"
HEALTHY_LABEL  = "PN"
DISEASE_LABEL  = "ALS"
#ALS_GENES      = ["SOD1", "C9orf72", "TARDBP"]
ALS_GENES      = ["SOD1", "C9orf72", "TARDBP", "FUS","UBQLN2", "OPTN", "TBK1", "VCP",
                   "HNRNPA1", "HNRNPA2B1", "NEK1", "SETX", "ANG", "PFN1", "SQSTM1"]



### Load & inspect data

In [4]:
adata = ad.read_h5ad("../counts_combined_filtered_BA4_sALS_PN.h5ad")

print(adata)
print("\ndisease labels:", adata.obs[CONDITION_COL].value_counts().to_dict())
print("cell types    :", adata.obs.get("CellType", adata.obs.columns).value_counts().head(10))

AnnData object with n_obs × n_vars = 112014 × 22832
    obs: 'Sample_ID', 'Donor', 'Region', 'Sex', 'Condition', 'Group', 'C9_pos', 'CellClass', 'CellType', 'SubType', 'full_label', 'DGE_Group', 'Bakken_M1', 'data_merge_id', 'data_sample_id', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'Cellstates_LVL1', 'Cellstates_LVL2', 'Cellstates_LVL3', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'n_genes', 'split'
    var: 'Biotype', 'Chromosome', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'ENSID', 'mt', 'n_cells', 'biotype'

disease labels: {'ALS': 66960, 'PN': 45054}
cell types    : CellType
Oligo     19043
Astro     13178
L2_L3     11871
OPC        9416
L3_L5      9331
L4_L6      7512
L5_L6      6608
5HT3aR     6034
PV         5595
L6         4848
Name: count, dtype: int64


In [5]:
# Restore raw counts if needed
if adata.raw is not None:
    adata = adata.raw.to_adata()
    print("Restored raw counts from adata.raw")

# Confirm ALS genes exist in dataset
missing = [g for g in ALS_GENES if g not in adata.var_names]
found   = [g for g in ALS_GENES if g in adata.var_names]
print(f"\nGenes found   : {len(found)}/{len(ALS_GENES)} → {found}")
print(f"Genes missing : {missing}")


Genes found   : 15/15 → ['SOD1', 'C9orf72', 'TARDBP', 'FUS', 'UBQLN2', 'OPTN', 'TBK1', 'VCP', 'HNRNPA1', 'HNRNPA2B1', 'NEK1', 'SETX', 'ANG', 'PFN1', 'SQSTM1']
Genes missing : []


### Split into healthy and disease

In [6]:
adata_healthy, adata_disease = split_by_condition(
    adata,
    condition_col = CONDITION_COL,
    healthy_label = HEALTHY_LABEL,
    disease_label = DISEASE_LABEL,
)

print(f"Healthy cells : {adata_healthy.n_obs:,}")
print(f"Disease cells : {adata_disease.n_obs:,}")

INFO:src.perturbation_v3:Split: 45054 healthy (PN) | 66960 disease (ALS)
2026-03-10 10:04:42,414 - INFO:src.perturbation_v3:Split: 45054 healthy (PN) | 66960 disease (ALS)


Healthy cells : 45,054
Disease cells : 66,960


### Sample from each dataframe and then merge them together

For simplicity, stratified sampling approach is used here. 

In [7]:
adata_disease_sub = _subsample_als(adata_disease, n_cells=2000)
adata_healthy_sub = _subsample_als(adata_healthy, n_cells=2000)

adata_final_sub = ad.concat(
    [adata_disease_sub, adata_healthy_sub],
    join="inner",    # inner join since var is identical in both
    merge="same",
)
print(f"Final dataset: {adata_final_sub.n_obs} cells × {adata_final_sub.n_vars} genes")

Original: 66960 cells × 22832 genes
Subsampling to 2000 cells total → 105 per cell type across 19 types:


Final : 1995 cells × 22832 genes
obs   : ['Sample_ID', 'Donor', 'Region', 'Sex', 'Condition', 'Group', 'C9_pos', 'CellClass', 'CellType', 'SubType', 'full_label', 'DGE_Group', 'Bakken_M1', 'data_merge_id', 'data_sample_id', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'Cellstates_LVL1', 'Cellstates_LVL2', 'Cellstates_LVL3', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'n_genes', 'split']
var   : ['Biotype', 'Chromosome', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'ENSID', 'mt', 'n_cells', 'biotype']
Original: 45054 cells × 22832 genes
Subsampling to 2000 cells total → 105 per cell type across 19 types:


Final : 1934 cells × 22832 genes
obs   : ['Sample_ID', 'Donor', 'Region', 'Sex', 'Condition', 'Group', 'C9_pos', 'CellClass', 'CellType', 'SubType'

### Initialize GeneFormer Model

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_config = GeneformerConfig(
    model_name="gf-12L-95M-i4096",
    batch_size=16,
    device=device,
    nproc=8, # Number of parallel processes for data loading
)

geneformer = Geneformer(configurer=model_config)
print("GeneFormer model loaded successfully.")

2026-03-10 10:04:50,380 - WARNING:helical.models.geneformer.geneformer_config:Setting model to gf-12L-38M-i4096. Model name gf-12L-95M-i4096 is deprecated. Please use the new name going forward to avoid code breakages.Geneformer models have been renamed to better reflect their size.


Using device: cpu


INFO:helical.models.geneformer.model:Model finished initializing.
2026-03-10 10:04:51,788 - INFO:helical.models.geneformer.model:Model finished initializing.
INFO:helical.models.geneformer.model:'gf-12L-38M-i4096' model is in 'eval' mode, on device 'cpu' with embedding mode 'cell'.
2026-03-10 10:04:51,789 - INFO:helical.models.geneformer.model:'gf-12L-38M-i4096' model is in 'eval' mode, on device 'cpu' with embedding mode 'cell'.


GeneFormer model loaded successfully.


## Run Perturbations

We run the pipeline on a subsampled dataset of healthy and diseased cells
1. **Healthy (PN) cells** — what does perturbing these genes do to normal cells?
2. **Disease (ALS) cells** — what does perturbing these genes do to already-diseased cells?

This contrast is the key scientific question of Task 2.

### Select only the HVGs for perturbation

In [9]:
# Save raw counts before any normalization
adata_final_sub.layers["raw_counts"] = adata_final_sub.X.copy()

# Do HVG selection on normalized data (normalization needed to find HVGs)
sc.pp.normalize_total(adata_final_sub, target_sum=1e4)
sc.pp.log1p(adata_final_sub)
sc.pp.highly_variable_genes(adata_final_sub, n_top_genes=2000)

# Build gene mask: HVGs + forced ALS genes
hvg_mask      = adata_final_sub.var["highly_variable"]
als_gene_mask = adata_final_sub.var_names.isin(ALS_GENES)

# Combine masks — union of HVGs and ALS genes
combined_mask = hvg_mask | als_gene_mask

# ALS gene in HVG
print("ALS genes in HVGs.      :", adata_final_sub.var_names[hvg_mask & als_gene_mask].tolist())
print(f"HVGs selected          : {hvg_mask.sum()}")
print(f"ALS genes forced in    : {als_gene_mask.sum()}")
print(f"Total genes after union: {combined_mask.sum()}")

# Single slice — preserves all var and obs exactly
adata_hvg = adata_final_sub[:, combined_mask].copy()

# Restore raw counts
adata_hvg.X = adata_hvg.layers["raw_counts"]

# Clean up layers and uns to keep AnnData tidy
adata_hvg = _clean_adata(adata_hvg)

# Verify
assert set(adata_final_sub.obs.columns) == set(adata_hvg.obs.columns)
assert set(adata_final_sub.var.columns) == set(adata_hvg.var.columns)
print(f"\nFinal: {adata_hvg.n_obs} cells × {adata_hvg.n_vars} genes")
print(f"var columns: {adata_hvg.var.columns.tolist()}")

ALS genes in HVGs.      : ['PFN1']
HVGs selected          : 2000
ALS genes forced in    : 15
Total genes after union: 2014

Final: 3929 cells × 2014 genes
var columns: ['Biotype', 'Chromosome', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'ENSID', 'mt', 'n_cells', 'biotype', 'highly_variable', 'means', 'dispersions', 'dispersions_norm']


### Run on full dataset at once (healthy and disease)

In [13]:
print("=" * 50)
print("Running perturbations on all cells...")
print("=" * 50)

results, labels, conditions = run_perturbation_pipeline(
                                adata         = adata_hvg,
                                gene_list     = ALS_GENES,
                                strength      = 2.0,
                                heterogeneity = 0.2,
                                modes         = MODES,
                                model         = geneformer,
                                            )

INFO:src.perturbation_v3:Starting perturbation pipeline.
2026-03-10 10:18:56,875 - INFO:src.perturbation_v3:Starting perturbation pipeline.
INFO:src.perturbation_v3:Genes: ['SOD1', 'C9orf72', 'TARDBP', 'FUS', 'UBQLN2', 'OPTN', 'TBK1', 'VCP', 'HNRNPA1', 'HNRNPA2B1', 'NEK1', 'SETX', 'ANG', 'PFN1', 'SQSTM1']
2026-03-10 10:18:56,877 - INFO:src.perturbation_v3:Genes: ['SOD1', 'C9orf72', 'TARDBP', 'FUS', 'UBQLN2', 'OPTN', 'TBK1', 'VCP', 'HNRNPA1', 'HNRNPA2B1', 'NEK1', 'SETX', 'ANG', 'PFN1', 'SQSTM1']
INFO:src.perturbation_v3:Modes: ['knockdown', 'knockup']
2026-03-10 10:18:56,878 - INFO:src.perturbation_v3:Modes: ['knockdown', 'knockup']
INFO:src.perturbation_v3:Computing baseline embeddings...
2026-03-10 10:18:56,884 - INFO:src.perturbation_v3:Computing baseline embeddings...
INFO:helical.models.geneformer.model:Processing data for Geneformer.
2026-03-10 10:18:56,885 - INFO:helical.models.geneformer.model:Processing data for Geneformer.


Running perturbations on all cells...


INFO:helical.utils.mapping:Mapped 2013 / 2014 genes to Ensembl IDs.
2026-03-10 10:18:57,423 - INFO:helical.utils.mapping:Mapped 2013 / 2014 genes to Ensembl IDs.
INFO:helical.models.geneformer.geneformer_tokenizer:AnnData object with n_obs × n_vars = 3929 × 2014
    obs: 'Sample_ID', 'Donor', 'Region', 'Sex', 'Condition', 'Group', 'C9_pos', 'CellClass', 'CellType', 'SubType', 'full_label', 'DGE_Group', 'Bakken_M1', 'data_merge_id', 'data_sample_id', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'Cellstates_LVL1', 'Cellstates_LVL2', 'Cellstates_LVL3', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'n_genes', 'split'
    var: 'Biotype', 'Chromosome', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'ENSID', 'mt', 'n_cells', 'biotype', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'index', 'ensembl_id', 'ensembl_id_collapsed' has no column attribute

  0%|          | 0/246 [00:00<?, ?it/s]

INFO:helical.models.geneformer.model:Finished getting embeddings.
2026-03-10 10:21:09,851 - INFO:helical.models.geneformer.model:Finished getting embeddings.
INFO:src.perturbation_v3:[1/30] Running knockdown_SOD1
2026-03-10 10:21:09,856 - INFO:src.perturbation_v3:[1/30] Running knockdown_SOD1
  self._set_arrayXarray(i, j, x)

2026-03-10 10:21:09,860 - WARNING:py.warnings:/Users/aseelawdeh/Documents/helical/als_perturbation/helical/lib/python3.11/site-packages/scipy/sparse/_index.py:151: SparseEfficiencyWarning: Changing the sparsity structure of a csc_matrix is expensive. lil_matrix is more efficient.
  self._set_arrayXarray(i, j, x)

INFO:src.perturbation_v3:  ✓ knockdown_SOD1 | gene=SOD1 | col_idx=842 | strength=2.00 | heterogeneity=0.20
2026-03-10 10:21:09,875 - INFO:src.perturbation_v3:  ✓ knockdown_SOD1 | gene=SOD1 | col_idx=842 | strength=2.00 | heterogeneity=0.20
INFO:src.perturbation_v3:[2/30] Running knockdown_C9orf72
2026-03-10 10:21:09,876 - INFO:src.perturbation_v3:[2/30] R

...

INFO:helical.models.geneformer.model:Finished getting embeddings.
2026-03-10 11:24:29,671 - INFO:helical.models.geneformer.model:Finished getting embeddings.
INFO:src.perturbation_v3:Running perturbation knockup_PFN1
2026-03-10 11:24:29,672 - INFO:src.perturbation_v3:Running perturbation knockup_PFN1
INFO:helical.models.geneformer.model:Processing data for Geneformer.
2026-03-10 11:24:29,672 - INFO:helical.models.geneformer.model:Processing data for Geneformer.
INFO:helical.utils.mapping:Mapped 2013 / 2014 genes to Ensembl IDs.
2026-03-10 11:24:30,179 - INFO:helical.utils.mapping:Mapped 2013 / 2014 genes to Ensembl IDs.
INFO:helical.models.geneformer.geneformer_tokenizer:AnnData object with n_obs × n_vars = 3929 × 2014
    obs: 'Sample_ID', 'Donor', 'Region', 'Sex', 'Condition', 'Group', 'C9_pos', 'CellClass', 'CellType', 'SubType', 'full_label', 'DGE_Group', 'Bakken_M1', 'data_merge_id', 'data_sample_id', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_cou

  0%|          | 0/246 [00:00<?, ?it/s]

INFO:helical.models.geneformer.model:Finished getting embeddings.
2026-03-10 11:26:52,132 - INFO:helical.models.geneformer.model:Finished getting embeddings.
INFO:src.perturbation_v3:Running perturbation knockup_SQSTM1
2026-03-10 11:26:52,133 - INFO:src.perturbation_v3:Running perturbation knockup_SQSTM1
INFO:helical.models.geneformer.model:Processing data for Geneformer.
2026-03-10 11:26:52,133 - INFO:helical.models.geneformer.model:Processing data for Geneformer.
INFO:helical.utils.mapping:Mapped 2013 / 2014 genes to Ensembl IDs.
2026-03-10 11:26:52,742 - INFO:helical.utils.mapping:Mapped 2013 / 2014 genes to Ensembl IDs.
INFO:helical.models.geneformer.geneformer_tokenizer:AnnData object with n_obs × n_vars = 3929 × 2014
    obs: 'Sample_ID', 'Donor', 'Region', 'Sex', 'Condition', 'Group', 'C9_pos', 'CellClass', 'CellType', 'SubType', 'full_label', 'DGE_Group', 'Bakken_M1', 'data_merge_id', 'data_sample_id', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total

  0%|          | 0/246 [00:00<?, ?it/s]

INFO:helical.models.geneformer.model:Finished getting embeddings.
2026-03-10 11:29:12,998 - INFO:helical.models.geneformer.model:Finished getting embeddings.
INFO:src.perturbation_v3:Perturbation pipeline complete. 2 conditions embedded.
2026-03-10 11:29:12,998 - INFO:src.perturbation_v3:Perturbation pipeline complete. 2 conditions embedded.


### Inspect embedding shapes

In [14]:
print("Results:")
baseline_emb = results["baseline"]
print(f"{'Baseline':35s}  shape: {baseline_emb.shape}")

for label, emb in results["perturbations"].items():
    print(f"  {label:35s}  {emb.shape}")


Results:
Baseline                             shape: (3929, 512)
  knockdown_SOD1                       (3929, 512)
  knockdown_C9orf72                    (3929, 512)
  knockdown_TARDBP                     (3929, 512)
  knockdown_FUS                        (3929, 512)
  knockdown_UBQLN2                     (3929, 512)
  knockdown_OPTN                       (3929, 512)
  knockdown_TBK1                       (3929, 512)
  knockdown_VCP                        (3929, 512)
  knockdown_HNRNPA1                    (3929, 512)
  knockdown_HNRNPA2B1                  (3929, 512)
  knockdown_NEK1                       (3929, 512)
  knockdown_SETX                       (3929, 512)
  knockdown_ANG                        (3929, 512)
  knockdown_PFN1                       (3929, 512)
  knockdown_SQSTM1                     (3929, 512)
  knockup_SOD1                         (3929, 512)
  knockup_C9orf72                      (3929, 512)
  knockup_TARDBP                       (3929, 512)
  knockup_FUS    

In [15]:
# Sanity check: cosine shift per gene
# We compute the cosine distance between the baseline embedding and each perturbation embedding,
# then rank the perturbations by how much they shift the embedding space.
# This is a quick way to see which perturbations have the biggest effect on the overall embedding space,
# which may correlate with biological relevance.

from sklearn.metrics.pairwise import cosine_distances
import pandas as pd

def compute_shifts(results: dict) -> pd.DataFrame:
    baseline = results["baseline"]
    rows = []
    for label, emb in results["perturbations"].items():
        mode, gene = label.split("_", 1)
        dist = cosine_distances(
            baseline.mean(axis=0, keepdims=True),
            emb.mean(axis=0, keepdims=True)
        )[0, 0]
        rows.append({"gene": gene, "mode": mode, "cosine_shift": dist})
    return pd.DataFrame(rows).sort_values("cosine_shift", ascending=False)

shifts = compute_shifts(results)

print("Top 5 perturbations by embedding shift:")
print(shifts.head())


Top 5 perturbations by embedding shift:
       gene       mode  cosine_shift
13     PFN1  knockdown      0.000161
28     PFN1    knockup      0.000155
3       FUS  knockdown      0.000146
23  HNRNPA1    knockup      0.000132
8   HNRNPA1  knockdown      0.000130


In [16]:
# Save results
np.savez("../outputs/task2/results_all_v2.npz", **results)
shifts.to_csv("../outputs/task2/shifts_all_v2.csv", index=False)
labels.to_csv("../outputs/task2/adata_all_celltypes_v2.csv", index=True)
adata_hvg.obs["Condition"].to_csv("../outputs/task2/adata_all_conditions_v2.csv", index=True)
print("Saved embeddings and shift summaries.")

Saved embeddings and shift summaries.


## Summary

| What was run | Cells | Conditions |
|---|---|---|
| Perturbations | PN + ALS | baseline + 15 genes × 2 modes = 31 |

All embeddings saved to `../data/` for interpretation in Notebook 03.

**Key outputs:**
- `results.npz` — embedding dict
- `adata_all_celltypes.csv` — cells labels 
- `shifts.csv` — cosine shift rankings

**Next:** Notebook 03 interprets the embedding space with UMAP,
clustering, and neighbourhood analysis.